In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
## Load data
df = pd.read_csv('processed_data/pcf_peak_table_ALL.csv')

df['date'] = pd.to_datetime(df['date'])
df = df.dropna(subset=['lek_id', 'date', 'peak_curvature'])

df.head()

,lek_id,date,s_peak,r_peak,g_peak,peak_prominence,peak_curvature,n_peaks,n_points
0,TalChhapar_TC,2012-10-01,1.471804,23.372757,1.248245,0.288691,3.674547,2,162
1,TalChhapar_TC,2012-10-01,2.614617,41.521015,1.003794,0.051697,1.322597,2,162
2,TalChhapar_TC,2014-02-01,1.521247,22.822810,1.084430,0.111816,2.960623,1,193
3,TalChhapar_TC,2015-11-01,1.530353,24.747625,1.126736,0.177206,1.585560,2,127
4,TalChhapar_TC,2015-11-01,2.805647,45.370646,0.987379,0.048893,1.394720,2,127


In [3]:
## Create consistent integer encodings for sites and years
site_lookup = {lek: i+1 for i,lek in enumerate(sorted(df['lek_id'].unique()))}
df['site_id'] = df['lek_id'].map(site_lookup)

unique_dates = sorted(df['date'].unique())
time_lookup = {d: i+1 for i,d in enumerate(unique_dates)}
df['time_id'] = df['date'].map(time_lookup)

In [4]:
## Ready dataframe to model the number of peaks on the pcf curves
n_peaks_df = df.groupby(['lek_id', 'site_id', 'date', 'time_id'], as_index=False).agg(n_peaks=('n_peaks', 'first'), n_points=('n_points', 'first'),)

## Ready dataframe to model the curvature of peaks on the pcf curves
curvature_df = df.sort_values(['lek_id', 'date', 'r_peak']).copy()
curvature_df['peak_num'] = curvature_df.groupby(['lek_id', 'date']).cumcount() + 1

curvature_df['log_curvature'] = np.log(curvature_df['peak_curvature'])
curvature_df = curvature_df[['lek_id', 'site_id', 'date', 'time_id', 'n_points', 'n_peaks', 'peak_num', 
                             'r_peak', 's_peak', 'peak_prominence', 'peak_curvature', 'log_curvature']].reset_index(drop=True)

## Build curve_id mapping from lek_id × date
curves = curvature_df[['lek_id', 'date']].drop_duplicates().sort_values(['lek_id', 'date']).reset_index(drop=True)
curves['curve_id'] = np.arange(1, len(curves) + 1)

In [5]:
## Merge curve_id into both dataframes
curvature_df = curvature_df.merge(curves, on=['lek_id', 'date'], how='left')
n_peaks_df = n_peaks_df.merge(curves, on=['lek_id', 'date'], how='left')

## Standardize peak_num for nicer sampling / interpretation
curvature_df['peak_num_z'] = (curvature_df['peak_num'] - curvature_df['peak_num'].mean()) / curvature_df['peak_num'].std(ddof=0)

In [ ]:
## Model the number of peaks for each lek
n_peaks_mod = """
data {
  int<lower=1> n_points;
  int<lower=1> n_sites;
  array[N] int<lower=1, upper=S> site_id;
  array[N] int<lower=0> peak_count;
}

parameters {
  real alpha;
  vector[n_sites] site_eff;
  real<lower=0> phi;
}

model {
  alpha ~ normal(0, 2);
  site_eff ~ normal(0, 1);
  phi ~ exponential(1);

  for (n in 1:n_points) {
    real eta = alpha + site_eff[site_id[n]];
    peak_count[n] ~ neg_binomial_2_log(eta, phi);
  }
}

generated quantities {
  real ratio_2_1 = exp(site_eff[2] - site_eff[1]);
  real ratio_3_1 = exp(site_eff[3] - site_eff[1]);
  real ratio_3_2 = exp(site_eff[3] - site_eff[2]);

  // Also useful: posterior predictive draw (optional)
  // array[n_points] int y_rep;
  // for (n in 1:n_points) y_rep[n] = neg_binomial_2_log_rng(alpha + site_eff[site_id[n]], phi);
}
"""

data_peaks = {
    'n_points': len(n_peaks_df),
    'n_sites': int(n_peaks_df['site_id'].nunique()),
    'site_id': n_peaks_df['site_id'].astype(int).to_numpy(),
    'peak_count': n_peaks_df['n_peaks'].astype(int).to_numpy()
}

posterior = stan.build(n_peaks_mod, data=data)
fit = posterior.sample(num_chains=4, num_warmup=1000, num_samples=1000)

In [ ]:
curvature_mod = """
data {
  int<lower=1> N;                          // number of peaks (rows)
  int<lower=1> S;                          // number of sites/leks (e.g., 3)
  int<lower=1> C;                          // number of curves = unique (lek_id × date)

  array[N] int<lower=1, upper=S> site_id;  // site for each peak
  array[N] int<lower=1, upper=C> curve_id; // curve group for each peak

  vector[N] peak_num_z;                    // standardized peak number
  vector[N] log_curvature;                 // log(peak_curvature)
}

parameters {
  real alpha;                               // grand intercept (log scale)

  // Site-level fixed intercepts and optional slope deviations
  vector[S] site_int_raw;
  real<lower=0> sigma_site_int;

  real beta_peak;                           // global effect of peak number (expect < 0)

  vector[S] site_slope_raw;                 // site-specific slope deviations (optional)
  real<lower=0> sigma_site_slope;           // shrinkage for site slope deviations

  // Curve-level random intercepts and optional random slopes
  vector[C] curve_int_raw;
  real<lower=0> sigma_curve_int;

  // Observation noise on log scale
  real<lower=0> sigma;
}

transformed parameters {
  vector[S] site_int = site_int_raw * sigma_site_int;

  // site-specific slopes around global beta_peak
  vector[S] site_slope = beta_peak + site_slope_raw * sigma_site_slope;
}

model {
  // Priors (weakly informative)
  alpha ~ normal(0, 5);                     // wide on log scale

  site_int_raw ~ normal(0, 1);
  sigma_site_int ~ normal(0, 2);

  beta_peak ~ normal(0, 1);                 // with peak_num_z, slope ~ O(1)

  site_slope_raw ~ normal(0, 1);
  sigma_site_slope ~ normal(0, 1);

  sigma_curve_int ~ normal(0, 2);

  sigma ~ normal(0, 1);

  // Likelihood
  for (n in 1:N) {
    real mu = alpha + site_int[site_id[n]] + beta_peak * peak_num_z[n];

    log_curvature[n] ~ normal(mu, sigma);
  }
}

generated quantities {
  // Site contrasts on log scale (S assumed >=3; adjust if needed)
  real diff_int_2_1 = site_int[2] - site_int[1];
  real diff_int_3_1 = site_int[3] - site_int[1];
  real diff_int_3_2 = site_int[3] - site_int[2];

  // Geometric-mean ratios for curvature (multiplicative)
  real gmr_int_2_1 = exp(diff_int_2_1);
  real gmr_int_3_1 = exp(diff_int_3_1);
  real gmr_int_3_2 = exp(diff_int_3_2);

  // Peak-number slope per site (on standardized scale)
  // Interpretable as multiplicative change per +1 SD in peak_num
  vector[S] slope_mult;
  for (s in 1:S) slope_mult[s] = exp(site_slope[s]);
}
"""

data_curv = {
    'N': len(curv_df),
    'S': int(curv_df['site_id'].nunique()),
    'C': int(curv_df['curve_id'].nunique()),
    'site_id': curv_df['site_id'].astype(int).to_numpy(),
    'curve_id': curv_df['curve_id'].astype(int).to_numpy(),
    'peak_num_z': curv_df['peak_num_z'].astype(float).to_numpy(),
    'log_curvature': curv_df['log_curvature'].astype(float).to_numpy(),
}

posterior = stan.build(curvature_mod, data=data)
fit = posterior.sample(num_chains=4, num_warmup=1000, num_samples=1000)